# MLflow Metrics Dashboard

Ноутбук для запуска MLflow, оценки сегментатора талька и классификатора, просмотра метрик, confusion matrix и overlay/error examples.

In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    """Find repository root from known local path or parent directories."""
    known_roots = [Path(r'D:/Nornikel-2026-Shlif-Case')]
    start = (start or Path.cwd()).resolve()
    markers = [Path('configs'), Path('app')]
    for candidate in [*known_roots, start, *start.parents]:
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise FileNotFoundError('Project root not found. Set ROOT manually to D:/Nornikel-2026-Shlif-Case.')


ROOT = find_project_root()
sys.path.insert(0, str(ROOT))
import json
import pandas as pd
from IPython.display import Image, display

%cd {ROOT}

MLRUNS = ROOT / 'artifacts/mlruns'
MLRUNS.mkdir(parents=True, exist_ok=True)
ROOT


## MLflow UI

Запусти эту команду в отдельном терминале или раскомментируй cell magic ниже. Потом открой http://127.0.0.1:5000.

In [ ]:
# !mlflow ui --backend-store-uri artifacts/mlruns --host 127.0.0.1 --port 5000

## Talc Segmenter: Train / Evaluate

In [ ]:
# Build dataset from manual masks
!python -m scripts.build_talc_dataset

# Train with MLflow logging enabled in config
# !python -m scripts.train_talc_segmenter

# Evaluate checkpoint, save full train/val overlays, log metrics/artifacts to MLflow
!python -m scripts.evaluate_talc_segmenter --checkpoint artifacts/runs/talc_segmenter/best.pt --output-dir artifacts/evaluation/talc_segmenter --mlflow

In [ ]:
seg_summary = json.loads((ROOT / 'artifacts/evaluation/talc_segmenter/summary.json').read_text(encoding='utf-8'))
pd.DataFrame(seg_summary).T

## Talc Segmenter Overlays

Цвета overlay: зелёный = ground truth only, красный = prediction only, жёлтый = overlap.

In [ ]:
for subset in ['train', 'val']:
    print(subset)
    for path in sorted((ROOT / f'artifacts/evaluation/talc_segmenter/{subset}/overlays').glob('*.jpg'))[:12]:
        display(Image(filename=str(path)))

## Classifier Evaluation

Обычный classifier baseline: метрики, confusion matrix и примеры ошибок.

In [ ]:
# Train classifier if needed:
# !python -m scripts.train_classifier

!python -m scripts.evaluate_classifier --checkpoint artifacts/runs/classifier_baseline/best.pt --output-dir artifacts/evaluation/classifier --mlflow

In [ ]:
cls_summary = json.loads((ROOT / 'artifacts/evaluation/classifier/summary.json').read_text(encoding='utf-8'))
display(pd.DataFrame(cls_summary).T)
display(pd.read_csv(ROOT / 'artifacts/evaluation/classifier/val/confusion.csv'))
error_sheet = ROOT / 'artifacts/evaluation/classifier/val/error_examples.jpg'
if error_sheet.exists():
    display(Image(filename=str(error_sheet)))

## Classifier With Talc-Mask Channel

Перед обучением 4-канального классификатора нужно сгенерировать маски талька для classification-изображений. RGB идёт как первые 3 канала, predicted talc mask как 4-й канал.

In [ ]:
# Generate talc-mask channel for classifier inputs
# !python -m scripts.predict_talc_masks checkpoint=artifacts/runs/talc_segmenter/best.pt output_dir=artifacts/predictions/talc_masks

# Train 4-channel classifier
# !python -m scripts.train_classifier --config configs/classifier/nornikel_classifier_talc_mask_channel.json

# Evaluate 4-channel classifier
# !python -m scripts.evaluate_classifier --checkpoint artifacts/runs/classifier_talc_mask_channel/best.pt --output-dir artifacts/evaluation/classifier_talc_mask_channel --mlflow